# Protein Design AI — Interview Live Demo

> **Pipeline:** ESM-2 Embedding → Surrogate MLP → Bayesian Optimisation → ProteinMPNN → RL REINFORCE

This notebook walks through the complete project end-to-end.  
Each section corresponds to a key competency for the **大分子 AI 演算法研究** role.

---
| Step | What we do | Why it matters |
|------|-----------|----------------|
| 1 | Generate / load protein sequences + fitness labels | Data pipeline |
| 2 | Extract ESM-2 embeddings (pre-trained protein LM) | Transfer learning |
| 3 | Train MLP surrogate model | Supervised regression |
| 4 | Bayesian Optimisation in latent space | Sample-efficient search |
| 5 | ProteinMPNN — structure-conditioned design | GNN, graph message passing |
| 6 | REINFORCE RL — sequence generation | Policy gradient, RL for biology |

## 0. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

# project imports
import sys
sys.path.insert(0, str(Path.cwd()))
from src.data_prep   import make_demo_data, describe_dataset
from src.embeddings  import ESM2Embedder
from src.predictor   import PredictorTrainer
from src.bayes_opt   import BayesianOptimizer
from src.protein_mpnn import ProteinMPNNTrainer
from src.rl_reinforce  import REINFORCETrainer, SequencePolicy, MultiObjectiveReward
from src.visualize   import plot_pipeline_results, plot_rl_training, plot_mpnn_training

Path('outputs').mkdir(exist_ok=True)
print('Setup complete ✓')

---
## 1. Data — Protein Sequences + Fitness Labels

We generate a synthetic dataset where the **fitness function** is defined as:

$$
y = 0.4 \cdot f_{\text{hydrophobic}} - 0.3 \cdot f_{\text{GP}} + 0.2 \cdot f_{\text{charged}} + \epsilon
$$

This mimics a real-world scenario where a wet-lab assay provides a scalar fitness score for each variant.

In [ ]:
seqs, labels = make_demo_data(n=100, seq_len=56, seed=42)
describe_dataset(seqs, labels)

# show a few examples
print('\nSample sequences:')
for i in range(3):
    print(f'  {seqs[i][:30]}...  fitness={labels[i]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(labels.numpy(), bins=20, color='steelblue', edgecolor='white')
ax.set_xlabel('Fitness Score')
ax.set_ylabel('Count')
ax.set_title('Fitness Distribution (Demo Data)')
plt.tight_layout()
plt.show()

---
## 2. ESM-2 Protein Language Model Embeddings

We use **ESM-2** (`facebook/esm2_t6_8M_UR50D`, 8M params) as a frozen feature extractor.

**Why ESM-2?**
- Pre-trained on 250M protein sequences via Masked Language Modelling (MLM)
- Captures evolutionary co-variation and structural constraints implicitly
- Zero-shot transfer: no labelled data needed for the embedding step

**Mean pooling** over sequence tokens → fixed 320-dim vector per sequence:
$$
\mathbf{z} = \frac{\sum_{t=1}^{L} m_t \cdot \mathbf{h}_t}{\sum_{t=1}^{L} m_t}
$$
where $m_t \in \{0,1\}$ is the attention mask.

In [ ]:
embedder = ESM2Embedder(model_size='8M', batch_size=16)
X = embedder.transform(seqs)       # shape: (100, 320)
print(f'Embedding matrix shape: {X.shape}')  # (100, 320)

In [ ]:
# Visualise embedding structure with PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels.numpy(), cmap='RdYlGn', s=40, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Fitness')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('ESM-2 Embeddings — PCA 2D (coloured by fitness)')
plt.tight_layout()
plt.show()
print(f'\nTotal variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')

---
## 3. Surrogate Model — Neural Network Fitness Predictor

Architecture:
```
ESM-2 embedding (320)
    → Linear(320, 256) → LayerNorm → ReLU → Dropout(0.2)
    → Linear(256, 64)  → LayerNorm → ReLU → Dropout(0.2)
    → Linear(64, 1)    → scalar fitness
```

Trained with **AdamW** + **MSE loss** to predict fitness from embedding.

In [ ]:
y = labels.numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

trainer = PredictorTrainer(input_dim=embedder.embed_dim)
trainer.fit(X_tr, y_tr, epochs=100, lr=1e-3, print_every=25)

print()
metrics = trainer.evaluate(X_te, y_te)
y_pred  = trainer.predict(X_te)
trainer.save('outputs/predictor_esm2.pt')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Loss curve
axes[0].plot(trainer.train_losses, color='royalblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].set_yscale('log')

# Predicted vs actual
vmin, vmax = min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())
axes[1].scatter(y_te, y_pred, alpha=0.7, color='tomato', s=35)
axes[1].plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)
axes[1].set_xlabel('True Fitness')
axes[1].set_ylabel('Predicted Fitness')
axes[1].set_title(f'Surrogate Prediction  (Pearson r={metrics["pearson_r"]:.3f})')

plt.tight_layout()
plt.show()

---
## 4. Bayesian Optimisation in Latent Space

**Key idea:** We cannot query the real wet-lab for every sequence.  
BO uses a GP surrogate + acquisition function to decide *where to query next*.

$$
\alpha_{\text{LogEI}}(x) = \log \mathbb{E}\bigl[\max(f(x) - f^*, 0)\bigr]
$$

**Why PCA first?**  
GP covariance is ill-conditioned in 320-D. PCA to 8-D keeps 81.9% variance and makes the GP numerically stable.

```
Embedding (320D) → PCA (8D) → GP fit → LogEI acquisition → argmax → inverse PCA → oracle query
```

In [ ]:
def oracle(X_scaled):
    return trainer.predict(trainer.scaler.inverse_transform(X_scaled))

opt = BayesianOptimizer(oracle_fn=oracle, n_pca_dims=8)
X_init_scaled = trainer.scaler.transform(X_tr)[:20]
opt.initialize(X_init_scaled, y_tr[:20])

bo_curve = opt.run(n_iter=20)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(bo_curve) + 1), bo_curve, marker='o', color='darkorange', linewidth=2)
ax.axhline(y_tr.max(), color='gray', linestyle='--', label=f'Initial best: {y_tr.max():.4f}')
ax.set_xlabel('BO Iteration')
ax.set_ylabel('Best Fitness Found')
ax.set_title('Bayesian Optimisation — Best Fitness over Iterations')
ax.legend()
plt.tight_layout()
plt.show()

improvement = (bo_curve[-1] - bo_curve[0]) / abs(bo_curve[0]) * 100
print(f'Improvement over 20 BO iterations: {bo_curve[0]:.4f} → {bo_curve[-1]:.4f}  ({improvement:+.1f}%)')

---
## 5. ProteinMPNN — Structure-Conditioned Sequence Design

**ProteinMPNN** designs amino-acid sequences given a protein backbone structure (Cα coordinates).

**Graph construction:**
- Nodes = residues, features = dihedral angles (sinusoidal, 16-D)
- Edges = k-NN (k=8) on Cα coordinates, features = relative distances + directions (19-D)

**Message passing** (L layers):
$$
h_v^{(l+1)} = \text{LN}\bigl(h_v^{(l)} + \text{ReLU}(W_O \cdot \textstyle\sum_{u \in \mathcal{N}(v)} \phi(h_v^{(l)}, h_u^{(l)}, e_{vu}))\bigr)
$$

Trained with **cross-entropy** to predict the correct amino acid at each position.

In [ ]:
mpnn_trainer = ProteinMPNNTrainer(hidden_dim=64, num_layers=2)
mpnn_losses, mpnn_seqs = mpnn_trainer.train_demo(
    seq_len=30, n_steps=40, lr=3e-4
)
plot_mpnn_training(mpnn_losses, 'outputs/mpnn_loss_demo.png')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mpnn_losses, color='purple', linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('ProteinMPNN — Training Loss')
plt.tight_layout()
plt.show()

print('\nSample designed sequences:')
for seq in mpnn_seqs:
    print(' ', seq)

---
## 6. REINFORCE — RL Policy for Sequence Generation

**Goal:** Learn a generative policy $\pi_\theta$ that samples sequences with high fitness.

**Architecture:** LSTM autoregressive sequence generator  
**State:** previously generated tokens  
**Action:** next amino acid token (vocab = 20 AA + START)  

**REINFORCE update:**
$$
\nabla_\theta J(\theta) = \mathbb{E}_\pi \bigl[ \nabla_\theta \log \pi_\theta(a_t | s_t) \cdot G_t \bigr]
$$

**Multi-objective reward:**
$$
R = w_{\text{stability}} \cdot r_{\text{stab}} + w_{\text{hydrophobic}} \cdot r_{\text{hydro}} + w_{\text{charged}} \cdot r_{\text{charge}}
$$

In [ ]:
import torch
from src.predictor import StabilityPredictor

# Use trained surrogate as fitness oracle for RL
def stability_oracle(sequences):
    """Embed sequences with ESM-2, predict fitness with surrogate MLP."""
    X_emb = embedder.transform(sequences)
    return trainer.predict(X_emb)

policy = SequencePolicy(seq_len=30, hidden_dim=128)
reward_fn = MultiObjectiveReward(
    oracle_fn=stability_oracle,
    w_stability=0.6, w_hydrophobic=0.3, w_charged=0.1
)
rl_trainer = REINFORCETrainer(policy, reward_fn, lr=5e-4)
rl_rewards = rl_trainer.run(n_episodes=25, batch_size=8)

In [ ]:
# Smooth reward curve
def smooth(arr, w=5):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rl_rewards, alpha=0.3, color='seagreen', label='Raw reward')
if len(rl_rewards) >= 5:
    ax.plot(range(4, len(rl_rewards)), smooth(rl_rewards), 
            color='seagreen', linewidth=2, label='Smoothed (w=5)')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean Reward')
ax.set_title('REINFORCE — Multi-Objective Reward over Episodes')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Starting reward: {rl_rewards[0]:.4f}')
print(f'Final   reward:  {rl_rewards[-1]:.4f}')
print(f'Improvement:     {rl_rewards[-1] - rl_rewards[0]:+.4f}')

---
## 7. Summary

| Component | Status | Key Detail |
|---|---|---|
| ESM-2 Embedding | ✅ | 320-D, mean-pooled, pre-trained on 250M seqs |
| Surrogate MLP | ✅ | LayerNorm + Dropout, trained on 80 sequences |
| Bayesian Optimisation | ✅ | GP + LogEI, PCA-8D, +16% fitness in 20 iters |
| ProteinMPNN | ✅ | k-NN Cα graph, scatter-add message passing |
| REINFORCE RL | ✅ | LSTM policy, teacher-forcing log-prob, multi-obj reward |

### Potential Extensions (for discussion)
- **Real wet-lab loop:** replace demo oracle with actual assay measurements
- **Larger ESM-2** (`esm2_t33_650M_UR50D`) for better representations  
- **Batched BO** (qEI) to propose multiple sequences per round  
- **PPO / SAC** instead of REINFORCE for more stable RL training  
- **FoldSeek / AlphaFold2** for structure-based fitness prediction

In [ ]:
import os
print('Output files:')
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}')
    print(f'  outputs/{f}  ({size/1024:.1f} KB)')